# Trendyol-LLM koc olcumu

`Trendyol/Trendyol-LLM-7B-chat-v4.1.0` modelini ucretsiz Colab T4'te vLLM ile
OpenAI-uyumlu bir uc nokta olarak acar, NEXORA API'sinin **gercek** koc prompt'unu
atar, sonucu olcer.

**Bu bir olcum kosumu, demo altyapisi degil.** Juri demosunun varsayilani canned
Turkce fallback olarak kalir (`docs/DEMO.md`). Colab oturumu ~90 dk bosta ve ~12
saatte oler, tunel URL'si her acilista degisir - buna bagli bir demo kurma.

## Neden

`docs/ARCHITECTURE.md` "LLM boundary" ve `docs/building-blocks/04-llm-coach.md`
olctu: `router.huggingface.co` 143 model sunuyor, hicbiri Trendyol degil. Urun
dokumani Trendyol-LLM vaat ediyor, mimari dokumani "saglanmiyor" diyor, arada
veri yok. Bu defter o veriyi uretir.

## Repo'dan buraya ne geliyor

Sadece iki metin: `apps/api/src/coach/index.ts`'teki `SYSTEM_PROMPT` ve
`buildUserPrompt` ciktisi. Ikisi de asagida birebir kopyali. Veritabani yok,
gercek kullanici yok, URL yok.

## Sonra ne olacak

Son hucrenin bastigi tabloyu `docs/building-blocks/04-llm-coach.md`'ye,
`Qwen/Qwen3-8B` olcumunun yanina tarihli bir not olarak yaz. Asil teslimat o.

> **Runtime > Change runtime type > T4 GPU.** Commit'ten once
> `Runtime > Restart and clear all outputs` - tunel URL'si ve api key
> ciktida kalmasin.

## 1 - Ortam

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm requests

## 2 - Sunucu

7B fp16 ~15 GB; T4'un 15 GB VRAM'ine KV cache ile birlikte sigmaz, o yuzden
bitsandbytes 4-bit. `--dtype half` **zorunlu**: T4 Turing mimarisi, bf16 yok.
`--max-model-len 2048` KV cache'i kucuk tutar - prompt birkac yuz token,
`max_tokens` 900.

Ilk acilis 7B agirligini indirip quantize eder: ~5-10 dk.

In [ ]:
import os, secrets, subprocess, time, requests

MODEL = "Trendyol/Trendyol-LLM-7B-chat-v4.1.0"
API_KEY = "nexora-" + secrets.token_hex(8)
PORT = 8000
LOCAL = f"http://127.0.0.1:{PORT}/v1"

ARGS = [
    "vllm", "serve", MODEL,
    "--quantization", "bitsandbytes",
    # ponytail: vLLM >= 0.9 infers the loader from --quantization and rejects
    # this flag. Drop the next line if the log says "unrecognized arguments".
    "--load-format", "bitsandbytes",
    "--dtype", "half",
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.92",
    "--api-key", API_KEY,
    "--port", str(PORT),
]

log = open("vllm.log", "w")
server = subprocess.Popen(ARGS, stdout=log, stderr=subprocess.STDOUT)

# Poll /health instead of sleeping a fixed amount: quantization time varies.
started = time.time()
deadline = started + 20 * 60
while time.time() < deadline:
    if server.poll() is not None:
        print("vLLM died. Last 40 log lines:\n")
        print("".join(open("vllm.log").readlines()[-40:]))
        raise SystemExit(1)
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            print(f"up after {int(time.time() - started)}s")
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise SystemExit("vLLM did not become healthy in 20 minutes")

## 3 - Olcum

`SYSTEM_PROMPT` birebir `apps/api/src/coach/index.ts`'ten. Uc kullanici prompt'u
`buildUserPrompt`'un gercek ciktisi - `packages/score`'un uc fixture'i
(`docs/building-blocks/03-signals-score.md`: risky 38, balanced 80, productive 93)
`computeScore`'dan gecirilip kopyalandi.

Istek govdesi de birebir ayni: `temperature 0.3`, `max_tokens 900`, `stream false`.

Olcum tunelden degil, localhost'tan yapilir - tunel gecikmesi sayilari kirletmesin.

In [ ]:
SYSTEM_PROMPT = "\n".join([
    "Sen 13-18 yaş arası gençler için Türkçe yazan bir dijital denge koçusun.",
    "Teşhis koymazsın, etiketlemezsin, suçlamazsın; kısa ve uygulanabilir öneriler verirsin.",
    "Yalnızca tek bir JSON nesnesi döndür, başka hiçbir metin yazma:",
    '{"tips":["","",""],"task":{"title":"","steps":["",""],"eta_minutes":15},"share_text":""}',
    "Kurallar: tips tam 3 madde; steps 2-5 madde; eta_minutes 5-30 arası tam sayı;",
    "hiçbir metinde bağlantı adresi olmasın; share_text veliyle paylaşılabilecek tek cümle olsun.",
])

# buildUserPrompt() output for the three score fixtures, verbatim.
PROMPTS = {
    "risky (38)": '{"minutes":{"science":10,"entertainment":200,"harmful":40},"score":{"value":38,"reasons":["Zararlı veya manipülatif kategoride süre var; bunu bir yetişkinle konuşmak iyi olabilir.","Eğlence kategorisi sürenin çoğunu kaplıyor.","Kategori çeşitliliğin düşük; kısa bir keşif görevi dene."]},"completed_tasks":[]}',
    "balanced (80)": '{"minutes":{"science":40,"arts":15,"sports":20,"culture":10,"national_memory":5,"entertainment":120},"score":{"value":80,"reasons":["Eğlence kategorisi sürenin çoğunu kaplıyor.","Bilim, sanat, spor veya kültür kategorilerinde görünür bir payın var.","Birden fazla değerli kategoride zaman geçirmişsin."]},"completed_tasks":[]}',
    "productive (93)": '{"minutes":{"science":70,"arts":20,"sports":20,"culture":15,"entertainment":40},"score":{"value":93,"reasons":["Bilim, sanat, spor veya kültür kategorilerinde görünür bir payın var.","Birden fazla değerli kategoride zaman geçirmişsin.","Zararlı/manipülatif kategoride süre görünmüyor."]},"completed_tasks":[]}',
}

# packages/shared coachSchema, as plain asserts. z.strictObject => no extra keys.
def schema_error(c):
    if not isinstance(c, dict) or set(c) != {"tips", "task", "share_text"}:
        return "top-level keys"
    if not (isinstance(c["tips"], list) and len(c["tips"]) == 3):
        return "tips != 3"
    if any(not isinstance(t, str) or not t for t in c["tips"]):
        return "empty tip"
    t = c["task"]
    if not isinstance(t, dict) or set(t) != {"title", "steps", "eta_minutes"}:
        return "task keys"
    if not isinstance(t["title"], str) or not t["title"]:
        return "task.title"
    if not (isinstance(t["steps"], list) and 2 <= len(t["steps"]) <= 5):
        return "steps out of 2-5"
    if any(not isinstance(s, str) or not s for s in t["steps"]):
        return "empty step"
    if not isinstance(t["eta_minutes"], int) or isinstance(t["eta_minutes"], bool):
        return "eta not int"
    if not 5 <= t["eta_minutes"] <= 30:
        return "eta out of 5-30"
    if not isinstance(c["share_text"], str) or not c["share_text"]:
        return "share_text"
    return None

# coach/index.ts BANNED, verbatim.
BANNED = ["teşhis", "teshis", "depresyon tanısı", "depresyon tanisi", "bağımlısın",
          "bagimlisin", "kötü çocuk", "kotu cocuk", "http://", "https://"]

In [ ]:
import json, statistics, time, requests

N = 20  # calls per band

def ask(user_prompt):
    """One call, shaped exactly like apps/api/src/coach/index.ts modelCoach()."""
    began = time.time()
    r = requests.post(
        f"{LOCAL}/chat/completions",
        headers={"content-type": "application/json", "authorization": f"Bearer {API_KEY}"},
        json={
            "model": MODEL,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": 0.3,
            "max_tokens": 900,
            "stream": False,
        },
        timeout=20,  # the API's AbortSignal.timeout(20_000)
    )
    return time.time() - began, r

results = {}
for band, prompt in PROMPTS.items():
    lat, ok, fenced, banned, errs = [], 0, 0, 0, []
    for _ in range(N):
        try:
            seconds, r = ask(prompt)
        except requests.RequestException as e:
            errs.append(type(e).__name__)
            continue
        lat.append(seconds)
        if not r.ok:
            errs.append(f"HTTP {r.status_code}")
            continue
        text = (r.json().get("choices") or [{}])[0].get("message", {}).get("content") or ""
        # coach/index.ts: models wrap JSON in prose or ``` fences.
        start, end = text.find("{"), text.rfind("}")
        if start > 0 or (end != -1 and end < len(text.rstrip()) - 1):
            fenced += 1
        try:
            coach = json.loads(text[start:end + 1])
        except (ValueError, TypeError):
            errs.append("not JSON")
            continue
        why = schema_error(coach)
        if why:
            errs.append(why)
            continue
        if any(b in json.dumps(coach, ensure_ascii=False).lower() for b in BANNED):
            banned += 1
            errs.append("banned phrase")
            continue
        ok += 1
    results[band] = {
        "n": N, "ok": ok, "fenced": fenced, "banned": banned,
        "p50": statistics.median(lat) if lat else None,
        "p95": sorted(lat)[int(len(lat) * 0.95) - 1] if len(lat) >= 2 else None,
        "errors": errs,
    }
    print(band, results[band])

In [ ]:
from collections import Counter

print(f"{MODEL}  |  T4, bitsandbytes 4-bit, dtype=half  |  N={N} per band\n")
print(f"{'band':<18}{'schema ok':>11}{'p50 s':>9}{'p95 s':>9}{'fenced':>9}{'banned':>9}")
for band, r in results.items():
    p50 = f"{r['p50']:.1f}" if r["p50"] else "-"
    p95 = f"{r['p95']:.1f}" if r["p95"] else "-"
    print(f"{band:<18}{str(r['ok']) + '/' + str(r['n']):>11}{p50:>9}{p95:>9}{r['fenced']:>9}{r['banned']:>9}")

total_ok = sum(r["ok"] for r in results.values())
total_n = sum(r["n"] for r in results.values())
print(f"\noverall schema hit rate: {total_ok}/{total_n} = {total_ok / total_n:.0%}")
print("Qwen/Qwen3-8B via the HF router measured ~3/4 (docs/building-blocks/04-llm-coach.md).")
print("\nfailures:", Counter(e for r in results.values() for e in r["errors"]).most_common())

## 4 - Uctan uca (opsiyonel)

Sayilari aldiysan is bitti. API'yi gercekten bu modele baglamak istersen tunel ac.

In [ ]:
import re, subprocess, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

tunnel_log = open("cloudflared.log", "w")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public = None
deadline = time.time() + 60
while time.time() < deadline and not public:
    time.sleep(2)
    hit = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    public = hit.group(0) if hit else None
if not public:
    raise SystemExit("no tunnel url; see cloudflared.log")

print("Paste into .env at the repo root, then `bun run dev --filter nexora-api`:\n")
print(f"HF_BASE_URL={public}/v1")
print(f"HF_TOKEN={API_KEY}")
print(f"HF_MODEL_ID={MODEL}")

Dogrulama (`docs/DEMO.md` personasi `deniz`, aktif ve skoru olan):

```
GET /coach/recommendation   ->   200, X-Nexora-Coach: model, source: "model"
```

Sonra bu defterin calisma zamanini **durdur** ve ayni cagriyi tekrarla:

```
GET /coach/recommendation   ->   200, X-Nexora-Coach: fallback
```

Ikincisi asil testtir: yeni upstream dustugunde de fail-closed davraniyor mu.
20 saniyelik `AbortSignal.timeout` icinde donmeli.

Bittiginde `.env`'den uc satiri sil - bayat bir tunel URL'si her koc cagrisina
20 saniye ekler.